Environment

In [ ]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}
hypcfg = "/home/zanardi/Codes/ML/PyCOMET/pycomet/examples/gno/cg/physics/hypcfg.json"

In [ ]:
import sys
for lib in libs.values():
    sys.path.append(lib)
import os
os.environ.update({"HYPNET_CONFIGFILE": hypcfg})

Import libraries

In [ ]:
import json
import numpy as np

from scipy.linalg import expm
from matplotlib import pyplot as plt
from silx.io.dictdump import h5todict

from hypernet import const
from hypernet.apps import utils as hyputils
from pycomet.networks import LinearMEGraphNet

Initializing thermophysical models

In [ ]:
with open("./physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Numerical solution

In [ ]:
sol = h5todict(h5file='./physics/solution.hdf5')

In [ ]:
na, nm = [sol[f"X_{s}"] * sol['n'] for s in sp_th.keys()]

In [ ]:
# Compute source term
t, T = [sol[k] for k in ('t', 'T')]
n = np.hstack([na, nm])
omega = kin.function(t=t, y=[n, T])
dnmdt = omega[:,1:-1] / const.UNA
dnmdt = dnmdt.numpy().squeeze()

Analytical solution

In [ ]:
# Solution function
nm_int = lambda t, n0, k: np.matmul(expm(k*t), n0)

In [ ]:
# Construct the evolution matrix
kf, kb = 0.0, 0.0
for reaction in kin.reactions:
    kf = kf + reaction.fwd.k.numpy().squeeze()[0]
    kb = kb + reaction.bwd.k.numpy().squeeze()[0]
k = kb.T + kf
k = (k.T - np.diag(np.sum(k, axis=-1))) * na[0]
# Compute exact solution
nm_anal = np.vstack([nm_int(ti, nm[0], k) for ti in t])
# Compute source term
dnmdt_anal = np.einsum("ij,nj->ni",k,nm_anal) / const.UNA

In [ ]:
plt.semilogx(t.squeeze(), nm_anal, '-')
plt.semilogx(t.squeeze(), nm, '--')

Construct graph

In [ ]:
# Nodes
node_attr = nm[...,None]
# Edges
k = kf + kb.T
k_idx = np.where(k > 0)
edge_index = np.array(k_idx)
edge_attr = (k[k_idx] * na[0]).reshape(1,-1,1)
edge_attr = np.tile(edge_attr, (len(node_attr),1,1))

Load linear master equation graph model

In [ ]:
model = LinearMEGraphNet(static_edge_index=edge_index)

In [ ]:
dnmdt_pred = model([node_attr, edge_attr]) / const.UNA
dnmdt_pred = dnmdt_pred.numpy().squeeze()

Plot comparison

In [ ]:
plt.semilogx(t.squeeze(), dnmdt, '-')
plt.semilogx(t.squeeze(), dnmdt_pred, '--')
# plt.semilogx(t.squeeze(), dnmdt_anal, '--')